In [0]:
# ── CÉLULA 1: criar volumes ──────────────────────────────────────────
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.silver")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.gold")

print("Volumes OK")

In [0]:
# ── CÉLULA 2: configuração ───────────────────────────────────────────
BRONZE_PATH = "/Volumes/workspace/default/bronze/"
SILVER_PATH = "/Volumes/workspace/default/silver/"
GOLD_PATH   = "/Volumes/workspace/default/gold/"

print("Config OK")

In [0]:
# ── CÉLULA 3: leitura do Bronze ──────────────────────────────────────
df_bronze_clientes = spark.read.format("delta").load(f"{BRONZE_PATH}clientes/")
df_bronze_produtos  = spark.read.format("delta").load(f"{BRONZE_PATH}produtos/")
df_bronze_pedidos   = spark.read.format("delta").load(f"{BRONZE_PATH}pedidos/")

print(f"Bronze clientes: {df_bronze_clientes.count()}")
print(f"Bronze produtos: {df_bronze_produtos.count()}")
print(f"Bronze pedidos:  {df_bronze_pedidos.count()}")

In [0]:
# ── CÉLULA 4: Silver clientes ────────────────────────────────────────
from pyspark.sql.functions import col, trim, upper, current_timestamp

silver_clientes = df_bronze_clientes \
    .dropDuplicates(["cliente_id"]) \
    .filter(col("nome").isNotNull()) \
    .withColumn("nome",     trim(col("nome"))) \
    .withColumn("cidade",   trim(col("cidade"))) \
    .withColumn("segmento", upper(trim(col("segmento")))) \
    .withColumn("_silver_processado", current_timestamp())

silver_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}clientes/")

print(f"Silver clientes: {silver_clientes.count()} linhas")
display(silver_clientes)

In [0]:
# ── CÉLULA 5: Silver produtos ────────────────────────────────────────
silver_produtos = df_bronze_produtos \
    .dropDuplicates(["produto_id"]) \
    .filter(col("preco") > 0) \
    .withColumn("categoria", upper(trim(col("categoria")))) \
    .withColumn("nome",      trim(col("nome"))) \
    .withColumn("_silver_processado", current_timestamp())

silver_produtos.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}produtos/")

print(f"Silver produtos: {silver_produtos.count()} linhas")
display(silver_produtos)

In [0]:
# ── CÉLULA 6: Silver pedidos ─────────────────────────────────────────
from pyspark.sql.functions import to_date

silver_pedidos = df_bronze_pedidos \
    .dropDuplicates(["pedido_id"]) \
    .filter(col("valor_total") > 0) \
    .withColumn("data_pedido", to_date(col("data_pedido"))) \
    .withColumn("status", upper(trim(col("status")))) \
    .withColumn("_silver_processado", current_timestamp())

silver_pedidos.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("status") \
    .save(f"{SILVER_PATH}pedidos/")

print(f"Silver pedidos: {silver_pedidos.count()} linhas")
display(silver_pedidos)